# `palfitology` — Cluster Sync + Scientific Analysis

A single notebook that does two jobs:

1. **Sync** every artifact you need from the CEFCA cluster down to your laptop — fitted PA tables, per-band PA plots, summary mosaics, NaN-clipped cutouts, reconciliation tables, GALFIT outputs (once they exist), and the original catalog.
2. **Analyse** the resulting data set the way a working astrophysicist would: sample-level diagnostics first, then population-level distributions of position angle and morphology, then a per-object deep dive for the interesting outliers.

> **Repo:** [`nischal-acharya27/palfitology`](https://github.com/nischal-acharya27/palfitology) · **Cluster:** CEFCA `hpc-login` (SGE) · **Convention:** PA in degrees, wrapped to [0°, 180°); circular distance bounded in [0°, 90°].

### Section map

| # | Section | What it does |
|---|---|---|
| 0 | Configuration | All paths / hostnames live here. Edit once. |
| 1 | Sync from cluster | `rsync` everything needed; dry-run first. |
| 2 | Local inventory | Walk what landed, surface gaps. |
| 3 | Load tables | `PA_results.csv`, reconciliation, catalog, GALFIT (if present). |
| 4 | Sample-level diagnostics | Coverage, imputation rate, weak fallbacks, fit status. |
| 5 | PA — populations + cross-band consistency | Circular stats, per-band scatter vs consensus, per-band dispersion. |
| 6 | PA vs catalog (`pa_jplus`) | Reconciliation scatter, residual histogram, outlier list. |
| 7 | Morphology (GALFIT, when available) | Sérsic n, R_e, axis ratio, magnitude — pair plots + flags. |
| 8 | Per-object deep dive | One object: cutouts, PA overlay, radial profile, residual. |
| 9 | Export figures + summary table | Drop a `figures/` dir + `analysis_summary.csv` for the writeup. |


## 0 · Configuration

Everything that depends on *where* code, data, or accounts live is here. Edit this cell once and the rest of the notebook is parameterized.

> **Memory pointer.** Your cluster paths (from `reference_paths.md`) are: cloned repo at `~/palfitology/`, data tree at `~/PALFITology/`, latest fits in `~/PALFITology_old/fitted_pa_images/`. Username and host alias live in your `~/.ssh/config`; the defaults below assume an alias `cefca`.


In [1]:
from __future__ import annotations
import os, subprocess, shlex, json, math, warnings
from pathlib import Path
from datetime import datetime

# ---- cluster side ------------------------------------------------------------
CLUSTER_HOST          = "hpc-login.office.cefca.es"                       # ssh alias from ~/.ssh/config (preferred)
CLUSTER_USER          = "nacharya"                            # optional; "user@host" if no alias
CLUSTER_DATA_ROOT     = "~/PALFITology_old"               # base data dir on the cluster
CLUSTER_FITTED_DIR    = "~/PALFITology_old/fitted_pa_images"   # `fit-pa` outputs
CLUSTER_CUTOUTS_DIR   = "~/PALFITology_old/images"        # raw + clipped cutouts per object
CLUSTER_GALFIT_DIR    = "~/PALFITology_old/galfit_out"    # GALFIT run dir (may not exist yet)
CLUSTER_CATALOG_PATH  = "~/PALFITology_old/catalog.csv"   # input catalog used by the run

# ---- laptop side -------------------------------------------------------------
LOCAL_ROOT     = Path.home() / "Desktop/AntigravityProjects/palfitology/cluster_pulls"
LOCAL_FITTED   = LOCAL_ROOT / "fitted_pa_images"
LOCAL_CUTOUTS  = LOCAL_ROOT / "images"
LOCAL_GALFIT   = LOCAL_ROOT / "galfit_out"
LOCAL_CATALOG  = LOCAL_ROOT / "catalog.csv"
LOCAL_FIGURES  = Path.home() / "Desktop/AntigravityProjects/palfitology/figures"

# ---- analysis defaults -------------------------------------------------------
ALL_BANDS = ["uJAVA","J0378","J0395","J0410","J0430","gSDSS",
             "J0515","rSDSS","J0660","iSDSS","J0861","zSDSS"]
BROAD_BANDS  = ["uJAVA","gSDSS","rSDSS","iSDSS","zSDSS"]
NARROW_BANDS = [b for b in ALL_BANDS if b not in BROAD_BANDS]
PRIMARY_BAND = "rSDSS"     # the deep band; what `detect.py` seeds geometry from

# Quality cuts (loose — tune per science case)
MIN_NDATA            = 5         # photutils.isophote ndata floor
MAX_PA_ERR_DEG       = 20.0      # discard rows with pa_err above this
PA_CONSENSUS_TOL_DEG = 10.0      # "agrees with consensus" if circ-diff <= this

# Make the local dirs (idempotent)
for p in (LOCAL_ROOT, LOCAL_FITTED, LOCAL_CUTOUTS, LOCAL_GALFIT, LOCAL_FIGURES):
    p.mkdir(parents=True, exist_ok=True)

def cluster_target(remote_path: str) -> str:
    host = f"{CLUSTER_USER}@{CLUSTER_HOST}" if CLUSTER_USER else CLUSTER_HOST
    return f"{host}:{remote_path}"

print("Cluster host:        ", CLUSTER_HOST)
print("Local pull root:     ", LOCAL_ROOT)
print("Fitted-PA outputs →  ", LOCAL_FITTED)
print("Cutouts →            ", LOCAL_CUTOUTS)
print("GALFIT outputs →     ", LOCAL_GALFIT)


Cluster host:         hpc-login.office.cefca.es
Local pull root:      /Users/nacharya/Desktop/AntigravityProjects/palfitology/cluster_pulls
Fitted-PA outputs →   /Users/nacharya/Desktop/AntigravityProjects/palfitology/cluster_pulls/fitted_pa_images
Cutouts →             /Users/nacharya/Desktop/AntigravityProjects/palfitology/cluster_pulls/images
GALFIT outputs →      /Users/nacharya/Desktop/AntigravityProjects/palfitology/cluster_pulls/galfit_out


## 1 · Sync the cluster outputs to the laptop

We use `rsync -avzhP` for the heavy directories — it's incremental (skips files that haven't changed), resumable (`P` keeps partial transfers), and shows progress. Everything is wrapped in a small Python helper so we can dry-run first, then commit.

**Order matters:** small text artifacts first (CSV tables, catalog), then PNGs (summaries), then FITS bulk last — so even if the bulk fails or you ctrl-c, you still have the things you'll actually plot from.


In [2]:
def _run(cmd: list[str] | str, dry: bool = False) -> int:
    """Pretty-print a shell command and (optionally) run it. Returns exit code (0 on dry)."""
    line = cmd if isinstance(cmd, str) else " ".join(shlex.quote(c) for c in cmd)
    prefix = "[DRY] " if dry else "[RUN] "
    print(prefix + line)
    if dry:
        return 0
    return subprocess.call(cmd, shell=isinstance(cmd, str))

def rsync_pull(remote_subpath: str, local_dest: Path, *, include: list[str] | None = None,
               exclude: list[str] | None = None, dry: bool = True) -> int:
    """Pull `remote_subpath` from the cluster into `local_dest`.

    `include` / `exclude` are passed straight to rsync — useful for grabbing
    only PNGs and CSVs without dragging the heavy FITS tree along.
    """
    args = ["rsync", "-avzhP", "--partial"]
    for pat in (exclude or []):
        args += ["--exclude", pat]
    for pat in (include or []):
        args += ["--include", pat]
    if include:
        args += ["--exclude", "*"]    # include-list mode: include then exclude everything else
    args += [cluster_target(remote_subpath) + ("/" if not remote_subpath.endswith("/") else ""),
             str(local_dest) + "/"]
    return _run(args, dry=dry)

# Quick connectivity probe — fail loudly before launching a 30-minute rsync.
def cluster_ping() -> bool:
    rc = _run(["ssh", "-o", "BatchMode=yes", "-o", "ConnectTimeout=5",
               (f"{CLUSTER_USER}@{CLUSTER_HOST}" if CLUSTER_USER else CLUSTER_HOST),
               "echo OK && hostname && date"], dry=False)
    return rc == 0


### 1a · Dry-run the whole sync (recommended first pass)

Set `DRY = False` once you're happy with what `rsync` *would* do.


In [3]:
DRY = True   # flip to False after you've eyeballed the dry-run

# 1) Catalog (tiny; always pull)
_run(["scp", cluster_target(CLUSTER_CATALOG_PATH), str(LOCAL_CATALOG)], dry=DRY)

# 2) Fitted-PA outputs — CSVs + PNGs only, no need to pull the per-band cutouts
#    that already live under `images/`. The fitted dir is mostly diagnostic plots.
rsync_pull(CLUSTER_FITTED_DIR, LOCAL_FITTED,
           include=["*/", "*.csv", "*.png"], dry=DRY)

# 3) Clipped + raw cutouts (the bulk).  Both for full provenance.
#    Drop sky-coverage *.weight.fits if you have them; keep only science + clipped.
rsync_pull(CLUSTER_CUTOUTS_DIR, LOCAL_CUTOUTS,
           exclude=["*.weight.fits", "*.rms.fits", "psfs_*/"],   # drop PSFs unless you need them
           dry=DRY)

# 4) GALFIT outputs (may not exist yet — that's fine, rsync will say so)
rsync_pull(CLUSTER_GALFIT_DIR, LOCAL_GALFIT,
           include=["*/", "*.fits", "*.log", "*.galfit", "*.png"], dry=DRY)


[DRY] scp 'nacharya@hpc-login.office.cefca.es:~/PALFITology_old/catalog.csv' /Users/nacharya/Desktop/AntigravityProjects/palfitology/cluster_pulls/catalog.csv
[DRY] rsync -avzhP --partial --include '*/' --include '*.csv' --include '*.png' --exclude '*' 'nacharya@hpc-login.office.cefca.es:~/PALFITology_old/fitted_pa_images/' /Users/nacharya/Desktop/AntigravityProjects/palfitology/cluster_pulls/fitted_pa_images/
[DRY] rsync -avzhP --partial --exclude '*.weight.fits' --exclude '*.rms.fits' --exclude 'psfs_*/' 'nacharya@hpc-login.office.cefca.es:~/PALFITology_old/images/' /Users/nacharya/Desktop/AntigravityProjects/palfitology/cluster_pulls/images/
[DRY] rsync -avzhP --partial --include '*/' --include '*.fits' --include '*.log' --include '*.galfit' --include '*.png' --exclude '*' 'nacharya@hpc-login.office.cefca.es:~/PALFITology_old/galfit_out/' /Users/nacharya/Desktop/AntigravityProjects/palfitology/cluster_pulls/galfit_out/


0

### 1b · Optional: pull only what's *new*

After the first big sync you can re-run the same `rsync_pull` calls — they'll only copy files whose size or mtime changed, so a re-sync after another cluster job is cheap.

### 1c · Alternative path: `sshfs` mount

If you'd rather browse the cluster live (slower for repeated reads, but no copy):

```bash
sshfs cefca:PALFITology ~/cefca_mount -o reconnect,ServerAliveInterval=30
```


## 2 · Local inventory — what landed?

Before plotting anything, walk what's now on disk. This catches missing objects, half-finished syncs, and bands the cluster didn't process. The output is a small dataframe you can sort/filter.


In [4]:
import pandas as pd

def inventory_fitted(root: Path) -> pd.DataFrame:
    """One row per object: how many band PNGs, summary mosaic present?, PA_fits.csv present?"""
    rows = []
    if not root.exists():
        print(f"[inventory] {root} does not exist — sync first.")
        return pd.DataFrame()
    for obj_dir in sorted(p for p in root.iterdir() if p.is_dir() and p.name != "all_summaries"):
        band_pngs = sorted(obj_dir.glob("*_PA_fit.png"))
        summary   = obj_dir / f"{obj_dir.name}_summary.png"
        per_obj_csv = obj_dir / "PA_fits.csv"
        rows.append({
            "id":           obj_dir.name,
            "n_band_pngs":  len(band_pngs),
            "has_summary":  summary.exists(),
            "has_PA_fits":  per_obj_csv.exists(),
            "bands_present": ",".join(p.name.split("_PA_fit")[0] for p in band_pngs),
        })
    df = pd.DataFrame(rows)
    print(f"[inventory] {len(df)} object dirs under {root}")
    return df

inv = inventory_fitted(LOCAL_FITTED)
inv.head()


[inventory] 0 object dirs under /Users/nacharya/Desktop/AntigravityProjects/palfitology/cluster_pulls/fitted_pa_images


""


In [5]:
# Quick health check: who is short on bands?
if len(inv):
    short = inv[inv["n_band_pngs"] < 12]
    print(f"{len(short)} / {len(inv)} objects have fewer than 12 bands.")
    short.head(15)


## 3 · Load the master tables

Four sources of truth:

| File | Rows | Purpose |
|---|---|---|
| `fitted_pa_images/PA_results.csv` | one per *(id, band)* | per-band fit + status + cutout_source |
| `fitted_pa_images/PA_reconciliation.csv` | one per id | catalog `pa_jplus` vs our per-band PAs (wide) |
| `catalog.csv` | one per id | original input (ra, dec, mag, pa_jplus, etc.) |
| `galfit_out/<id>/fit.log` | one per id (parsed) | Sérsic n, R_e, axis ratio, mag, χ² |

The GALFIT table is built on the fly from per-object `fit.log` files (GALFIT writes them in its own format, not CSV).


In [6]:
def safe_read_csv(path: Path, **kw) -> pd.DataFrame:
    if not path.exists():
        print(f"[load] missing: {path}")
        return pd.DataFrame()
    df = pd.read_csv(path, **kw)
    print(f"[load] {path.name}: {len(df)} rows · {len(df.columns)} cols")
    return df

pa_results  = safe_read_csv(LOCAL_FITTED / "PA_results.csv")
reconcile   = safe_read_csv(LOCAL_FITTED / "PA_reconciliation.csv")
catalog     = safe_read_csv(LOCAL_CATALOG)

# Normalise types we'll need a lot.
for df in (pa_results, reconcile, catalog):
    if "id" in df.columns:
        df["id"] = df["id"].astype(str)
if "band" in pa_results.columns:
    pa_results["band"] = pd.Categorical(pa_results["band"], categories=ALL_BANDS, ordered=True)

pa_results.head()


[load] missing: /Users/nacharya/Desktop/AntigravityProjects/palfitology/cluster_pulls/fitted_pa_images/PA_results.csv
[load] missing: /Users/nacharya/Desktop/AntigravityProjects/palfitology/cluster_pulls/fitted_pa_images/PA_reconciliation.csv
[load] missing: /Users/nacharya/Desktop/AntigravityProjects/palfitology/cluster_pulls/catalog.csv


""


In [7]:
# ---- GALFIT log parser -------------------------------------------------------
# GALFIT fit.log lines we care about look like:
#   Sersic  : (x,y) = (50.32, 49.78)  mag = 18.234  Re = 4.51  n = 1.83  q = 0.62  PA = 47.3
#   Chi^2/nu = 1.045
# Very tolerant regex; fields the user does not have just stay NaN.
import re

_GAL_PAT = re.compile(
    r"Sersic.*?mag\s*=\s*([0-9.\-]+).*?Re\s*=\s*([0-9.\-]+).*?n\s*=\s*([0-9.\-]+).*?"
    r"q\s*=\s*([0-9.\-]+).*?PA\s*=\s*([0-9.\-]+)",
    re.DOTALL | re.IGNORECASE,
)
_CHI2_PAT = re.compile(r"Chi\^2/nu\s*=\s*([0-9.\-]+)", re.IGNORECASE)

def parse_galfit_log(log_path: Path) -> dict | None:
    try:
        txt = log_path.read_text(errors="ignore")
    except OSError:
        return None
    m = _GAL_PAT.search(txt)
    if not m:
        return None
    mag, re_pix, n, q, pa = (float(x) for x in m.groups())
    chi = _CHI2_PAT.search(txt)
    return {
        "galfit_mag":    mag,
        "galfit_Re_pix": re_pix,
        "galfit_n":      n,
        "galfit_q":      q,      # axis ratio b/a
        "galfit_PA":     pa % 180.0,
        "galfit_chi2nu": float(chi.group(1)) if chi else None,
    }

def build_galfit_table(root: Path) -> pd.DataFrame:
    rows = []
    if not root.exists():
        return pd.DataFrame()
    for obj_dir in sorted(p for p in root.iterdir() if p.is_dir()):
        log = obj_dir / "fit.log"
        rec = parse_galfit_log(log) if log.exists() else None
        if rec:
            rec["id"] = obj_dir.name
            rows.append(rec)
    df = pd.DataFrame(rows)
    print(f"[galfit] parsed {len(df)} fit.log files from {root}")
    return df

galfit = build_galfit_table(LOCAL_GALFIT)
galfit.head() if len(galfit) else "(no GALFIT runs yet — Section 7 will be skipped)"


[galfit] parsed 0 fit.log files from /Users/nacharya/Desktop/AntigravityProjects/palfitology/cluster_pulls/galfit_out


'(no GALFIT runs yet — Section 7 will be skipped)'

## 4 · Sample-level diagnostics — *is this dataset trustworthy?*

Before any science, you want to know:

1. **Coverage** — what fraction of expected (id × band) cells actually have a row?
2. **Status mix** — `ok`, `weak_fallback`, `imputed`, `missing`. Imputation is fine for plotting but a red flag at >~10% of the sample.
3. **`cutout_source` mix** — for V0.6+ runs, what fraction used `clipped` vs `original`? Disagreement here can drive PA differences downstream.
4. **`pa_err` distribution** — long-tailed `pa_err` means a band consistently fails on this sample.


In [8]:
import matplotlib.pyplot as plt
import numpy as np

if len(pa_results):
    n_obj   = pa_results["id"].nunique()
    n_band  = pa_results["band"].nunique()
    expected = n_obj * n_band
    print(f"Sample: {n_obj} objects × {n_band} bands → {expected} expected rows; have {len(pa_results)}")
    print("\nStatus mix:")
    print(pa_results["status"].value_counts(dropna=False))
    if "cutout_source" in pa_results.columns:
        print("\ncutout_source mix:")
        print(pa_results["cutout_source"].value_counts(dropna=False))
    if "is_imputed" in pa_results.columns:
        impr = pa_results["is_imputed"].astype(bool).mean()
        print(f"\nImputation rate: {impr:.1%}")
    if "used_weak_fallback" in pa_results.columns:
        wfr = pa_results["used_weak_fallback"].astype(bool).mean()
        print(f"Weak-fallback rate: {wfr:.1%}")


In [9]:
# Per-band success rates — find the bands that fail systematically.
if len(pa_results) and "status" in pa_results.columns:
    tbl = (pa_results
           .assign(ok=lambda d: d["status"].eq("ok").astype(int))
           .groupby("band", observed=True)["ok"].mean()
           .reindex(ALL_BANDS))
    fig, ax = plt.subplots(figsize=(9, 3.2))
    ax.bar(tbl.index, tbl.values, color=["#3a86ff" if b in BROAD_BANDS else "#ffbe0b" for b in tbl.index])
    ax.set_ylim(0, 1.02); ax.set_ylabel("fraction status='ok'"); ax.set_title("Per-band fit success rate")
    ax.set_xticklabels(tbl.index, rotation=45, ha="right")
    for x, v in enumerate(tbl.values):
        if np.isfinite(v):
            ax.text(x, v + 0.01, f"{v:.0%}", ha="center", fontsize=8)
    fig.tight_layout(); plt.show()


In [10]:
# pa_err CDF per band — a band whose CDF lags is contributing scatter to consensus.
if len(pa_results) and "pa_err" in pa_results.columns:
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for band in ALL_BANDS:
        sub = pa_results.loc[pa_results["band"] == band, "pa_err"].dropna()
        sub = sub[sub > 0]
        if len(sub) < 3: continue
        x = np.sort(sub.values); y = np.arange(1, len(x)+1) / len(x)
        ax.plot(x, y, label=band, lw=1, alpha=0.85)
    ax.set_xscale("log"); ax.set_xlabel("pa_err [deg]"); ax.set_ylabel("CDF")
    ax.set_title("Per-band pa_err CDF (lower-right = better)")
    ax.axvline(MAX_PA_ERR_DEG, color="k", ls="--", lw=0.8, label=f"cut @ {MAX_PA_ERR_DEG}°")
    ax.legend(ncol=2, fontsize=8, loc="lower right"); ax.grid(alpha=0.3)
    fig.tight_layout(); plt.show()


## 5 · Position-angle populations + cross-band consistency

Two questions here:

1. **Is the PA distribution uniform?** For a magnitude-limited, randomly-oriented sample of disc galaxies it should be — non-uniformity often means a selection effect or a wrap-bug.
2. **Do the bands agree with each other?** If PA in one band consistently differs from the consensus by ~20°, that band's PSF or sky model is biasing the fit. Cross-band scatter is the cleanest internal check we have.

We use *circular* statistics throughout: PA is direction-only so we wrap to [0°, 180°) and compute the doubled-angle mean.


In [11]:
def circular_mean_deg(angles_deg: np.ndarray) -> float:
    """Doubled-angle (axial) circular mean in degrees, wrapped to [0, 180)."""
    a = np.deg2rad(np.asarray(angles_deg, float) * 2.0)
    a = a[np.isfinite(a)]
    if len(a) == 0: return np.nan
    return (np.rad2deg(np.arctan2(np.sin(a).mean(), np.cos(a).mean())) / 2.0) % 180.0

def circular_diff_deg(a, b) -> np.ndarray:
    d = np.abs(np.asarray(a, float) - np.asarray(b, float)) % 180.0
    return np.minimum(d, 180.0 - d)

# Per-object consensus from "ok" rows only (matches the pipeline's circular mean).
def consensus_pa(df: pd.DataFrame) -> pd.DataFrame:
    good = df[df.get("status", "ok").eq("ok")
              & (df.get("pa_err", 0) < MAX_PA_ERR_DEG)
              & (df.get("pa_err", 0) > 0)]
    g = good.groupby("id")["est_pa"].apply(circular_mean_deg).rename("consensus_pa")
    return g.reset_index()

if len(pa_results):
    consensus = consensus_pa(pa_results)
    print(f"Consensus PA built for {len(consensus)} / {pa_results['id'].nunique()} objects")
    consensus.head()


In [12]:
# 5a — Marginal PA histogram (consensus, then per-band overlay).
if len(pa_results):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].hist(consensus["consensus_pa"].dropna(), bins=np.linspace(0, 180, 19),
                 color="#264653", edgecolor="white")
    axes[0].set_xlim(0, 180); axes[0].set_xlabel("consensus PA [deg]"); axes[0].set_ylabel("objects")
    axes[0].set_title(f"Consensus PA (N={consensus['consensus_pa'].notna().sum()})")
    axes[0].axhline(consensus["consensus_pa"].notna().sum() / 18, color="grey", ls="--", lw=0.8,
                    label="uniform expectation")
    axes[0].legend(fontsize=8)

    # Per-band marginals as overlaid step histograms — quick "are all bands consistent?" eyeball.
    for band, color in zip([PRIMARY_BAND, "gSDSS", "iSDSS"], ["#e63946", "#2a9d8f", "#f4a261"]):
        sub = pa_results.loc[(pa_results["band"] == band) & pa_results["est_pa"].notna(), "est_pa"]
        axes[1].hist(sub, bins=np.linspace(0, 180, 19), histtype="step", lw=1.6, label=band, color=color)
    axes[1].set_xlim(0, 180); axes[1].set_xlabel("est_pa [deg]"); axes[1].set_title("Per-band marginals")
    axes[1].legend(fontsize=8)
    fig.tight_layout(); plt.show()


In [13]:
# 5b — Per-band scatter vs consensus.  Tight cloud along y=x → that band agrees with the group.
if len(pa_results):
    df = pa_results.merge(consensus, on="id")
    df = df[df["est_pa"].notna() & df["consensus_pa"].notna()]
    band_list = [b for b in ALL_BANDS if (df["band"] == b).any()]
    n = len(band_list); ncol = 4; nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.2*ncol, 3.0*nrow), sharex=True, sharey=True)
    for ax, band in zip(axes.flat, band_list):
        sub = df[df["band"] == band]
        ax.scatter(sub["consensus_pa"], sub["est_pa"], s=8, alpha=0.5, color="#1d3557")
        ax.plot([0,180],[0,180], color="grey", lw=0.6)
        diff = circular_diff_deg(sub["est_pa"], sub["consensus_pa"])
        rms = float(np.sqrt(np.nanmean(diff**2))) if len(diff) else np.nan
        ax.set_title(f"{band}  (N={len(sub)}, rms={rms:.1f}°)", fontsize=9)
        ax.set_xlim(0,180); ax.set_ylim(0,180)
    for ax in axes.flat[len(band_list):]:
        ax.set_visible(False)
    for ax in axes[-1]: ax.set_xlabel("consensus PA [deg]")
    for ax in axes[:,0]: ax.set_ylabel("band PA [deg]")
    fig.suptitle("Per-band PA vs consensus  ·  tight diagonal = agreement", y=1.01)
    fig.tight_layout(); plt.show()


In [14]:
# 5c — Per-band circular dispersion (rms of band-minus-consensus diff).
if len(pa_results):
    df = pa_results.merge(consensus, on="id")
    df["diff"] = circular_diff_deg(df["est_pa"], df["consensus_pa"])
    rms_by_band = (df.groupby("band", observed=True)["diff"]
                     .apply(lambda s: float(np.sqrt(np.nanmean(s**2))))
                     .reindex(ALL_BANDS))
    fig, ax = plt.subplots(figsize=(9, 3.2))
    ax.bar(rms_by_band.index, rms_by_band.values,
           color=["#3a86ff" if b in BROAD_BANDS else "#ffbe0b" for b in rms_by_band.index])
    ax.set_ylabel("circ-RMS(band − consensus) [deg]"); ax.set_title("Per-band PA dispersion")
    ax.set_xticklabels(rms_by_band.index, rotation=45, ha="right")
    fig.tight_layout(); plt.show()


## 6 · PA vs the J-PLUS catalog (`pa_jplus`)

If your consensus PA differs systematically from `pa_jplus`, either (a) the J-PLUS pipeline's PA convention has a 90° offset relative to photutils (would show as a y = x + 90° clustering), or (b) the J-PLUS PA is contaminated for that subset (deblending, edge-on cases). Both are worth knowing before publishing morphology.


In [15]:
def attach_catalog(consensus_df: pd.DataFrame, catalog_df: pd.DataFrame) -> pd.DataFrame:
    if not len(catalog_df) or "pa_jplus" not in catalog_df.columns:
        print("[reconcile] no pa_jplus column in catalog — skipping section 6.")
        return pd.DataFrame()
    cat = catalog_df.copy()
    cat["id"] = cat["id"].astype(str)
    merged = consensus_df.merge(cat[["id","pa_jplus"]], on="id", how="inner")
    merged["pa_jplus"] = merged["pa_jplus"].astype(float) % 180.0
    merged["consensus_pa"] = merged["consensus_pa"].astype(float) % 180.0
    merged["circ_diff"] = circular_diff_deg(merged["consensus_pa"], merged["pa_jplus"])
    return merged

if len(pa_results) and len(catalog):
    rec = attach_catalog(consensus, catalog)
    if len(rec):
        fig, axes = plt.subplots(1, 2, figsize=(11, 4))
        axes[0].scatter(rec["pa_jplus"], rec["consensus_pa"], s=10, alpha=0.5, color="#1d3557")
        axes[0].plot([0,180],[0,180], color="grey", lw=0.6, label="y = x")
        axes[0].plot([0,90],[90,180], color="#e63946", lw=0.6, ls="--", label="y = x + 90°")
        axes[0].plot([90,180],[0,90], color="#e63946", lw=0.6, ls="--")
        axes[0].set_xlim(0,180); axes[0].set_ylim(0,180)
        axes[0].set_xlabel("pa_jplus [deg]"); axes[0].set_ylabel("consensus PA [deg]")
        axes[0].legend(fontsize=8); axes[0].set_title("PA reconciliation")

        axes[1].hist(rec["circ_diff"], bins=np.linspace(0, 90, 19), color="#264653", edgecolor="white")
        axes[1].axvline(PA_CONSENSUS_TOL_DEG, color="#e63946", lw=1, label=f"tol = {PA_CONSENSUS_TOL_DEG}°")
        axes[1].set_xlabel("circular |Δ PA| [deg]"); axes[1].set_ylabel("objects")
        axes[1].set_title(f"Residuals  ·  median = {rec['circ_diff'].median():.1f}°")
        axes[1].legend(fontsize=8)
        fig.tight_layout(); plt.show()

        n_out = (rec["circ_diff"] > PA_CONSENSUS_TOL_DEG).sum()
        print(f"\n{n_out} / {len(rec)} objects disagree with catalog by > {PA_CONSENSUS_TOL_DEG}°")
        print(rec.nlargest(10, "circ_diff")[["id","pa_jplus","consensus_pa","circ_diff"]])


## 7 · GALFIT morphology (when the runs exist)

`palfitology.galfit` is currently a stub — the runs are produced externally (per-object `fit.log` files). This section is **opt-in**: if the local GALFIT tree is empty it just prints a note and skips.

Plots:

- **Sérsic index** (`n`) — distinguishes disc-like (`n ≲ 1.5`) from bulge-dominated (`n ≳ 3`).
- **Effective radius** (`Re_pix`) vs magnitude — Kormendy-style relation.
- **Axis ratio** (`q = b/a`) histogram — should rise toward `q = 1` for randomly oriented discs.
- **χ²/ν** vs `n` — pick out the dodgy fits.


In [16]:
if len(galfit):
    fig, axes = plt.subplots(2, 2, figsize=(10, 8))

    axes[0,0].hist(galfit["galfit_n"].clip(0, 8), bins=24, color="#264653", edgecolor="white")
    axes[0,0].set_xlabel("Sérsic n"); axes[0,0].set_ylabel("objects"); axes[0,0].set_title("Sérsic index")

    axes[0,1].scatter(galfit["galfit_mag"], galfit["galfit_Re_pix"], s=12, alpha=0.6, color="#1d3557")
    axes[0,1].set_yscale("log"); axes[0,1].set_xlabel("mag"); axes[0,1].set_ylabel("R_e [pix]")
    axes[0,1].set_title("Size–luminosity")

    axes[1,0].hist(galfit["galfit_q"].clip(0,1), bins=np.linspace(0,1,21),
                   color="#e76f51", edgecolor="white")
    axes[1,0].set_xlabel("axis ratio q = b/a"); axes[1,0].set_ylabel("objects")
    axes[1,0].set_title("Axis ratio")

    sc = axes[1,1].scatter(galfit["galfit_n"], galfit["galfit_chi2nu"],
                           c=galfit["galfit_q"], cmap="viridis", s=14)
    axes[1,1].set_xlabel("Sérsic n"); axes[1,1].set_ylabel("χ²/ν")
    axes[1,1].axhline(1.0, color="grey", lw=0.6); axes[1,1].axhline(2.0, color="#e63946", lw=0.6, ls="--")
    axes[1,1].set_title("Fit quality vs morphology")
    plt.colorbar(sc, ax=axes[1,1], label="q")
    fig.tight_layout(); plt.show()

    # Cross-check: does GALFIT PA agree with our consensus PA?
    if len(consensus):
        m = galfit.merge(consensus, on="id")
        m["pa_diff"] = circular_diff_deg(m["galfit_PA"], m["consensus_pa"])
        print(f"GALFIT vs consensus PA: median |Δ| = {m['pa_diff'].median():.2f}°, "
              f"90th %ile = {m['pa_diff'].quantile(0.9):.2f}°")
else:
    print("No GALFIT outputs found — skipping Section 7. "
          "Run GALFIT externally on the cluster and re-sync to populate this.")


No GALFIT outputs found — skipping Section 7. Run GALFIT externally on the cluster and re-sync to populate this.


## 8 · Per-object deep dive

Pick one ID — typically a PA outlier from §6 or a high-χ²/ν fit from §7 — and inspect everything we have on it:

1. The 3×4 summary mosaic (PNG, already rendered by the pipeline).
2. The reference-band raw cutout + its NaN-clipped sibling (FITS).
3. The reference-band ellipse overlay (regenerated from `PA_results.csv` for the chosen object).
4. The radial light profile from `photutils.isophote`.
5. (Optional) the GALFIT model + residual if a fit exists.

Set `OBJ_ID` and re-run the cells.


In [17]:
OBJ_ID = None   # e.g. "94261-12395" — or leave None to auto-pick the largest catalog disagreement
if OBJ_ID is None and 'rec' in globals() and len(rec):
    OBJ_ID = rec.sort_values("circ_diff", ascending=False).iloc[0]["id"]
print("Deep-diving:", OBJ_ID)


Deep-diving: None


In [18]:
from PIL import Image

def show_summary_png(obj_id: str):
    candidates = [
        LOCAL_FITTED / obj_id / f"{obj_id}_summary.png",
        LOCAL_FITTED / "all_summaries" / f"{obj_id}_summary.png",
    ]
    for p in candidates:
        if p.exists():
            img = Image.open(p)
            fig, ax = plt.subplots(figsize=(11, 8.5))
            ax.imshow(img); ax.set_axis_off(); ax.set_title(p.name)
            plt.show(); return
    print(f"No summary PNG found for {obj_id} in {LOCAL_FITTED}")

if OBJ_ID: show_summary_png(OBJ_ID)


In [19]:
# Raw vs clipped cutout in the reference band.
from astropy.io import fits
from astropy.visualization import ZScaleInterval, ImageNormalize, AsinhStretch

def _find_band_fits(obj_id: str, band: str) -> tuple[Path|None, Path|None]:
    """Return (raw_fits, clipped_fits) for this object/band, or (None, None) if absent."""
    obj_root = LOCAL_CUTOUTS / obj_id
    raw = clipped = None
    if not obj_root.exists():
        return None, None
    for sub in obj_root.iterdir():
        if sub.is_dir() and sub.name.startswith("fits_images_"):
            cand = sub / f"{band}_cutout.fits"
            if cand.exists(): raw = cand
        if sub.is_dir() and sub.name.startswith("clipped_cutouts_"):
            cand = sub / f"{band}_cutout.fits"
            if cand.exists(): clipped = cand
    return raw, clipped

def imshow_asinh(ax, arr, title=""):
    finite = arr[np.isfinite(arr)]
    if len(finite) == 0:
        ax.text(0.5, 0.5, "no data", transform=ax.transAxes, ha="center"); ax.set_axis_off(); return
    norm = ImageNormalize(finite, interval=ZScaleInterval(), stretch=AsinhStretch())
    ax.imshow(arr, origin="lower", cmap="gray_r", norm=norm)
    ax.set_title(title, fontsize=9); ax.set_xticks([]); ax.set_yticks([])

if OBJ_ID:
    raw, clipped = _find_band_fits(OBJ_ID, PRIMARY_BAND)
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    if raw:
        imshow_asinh(axes[0], fits.getdata(raw), f"{PRIMARY_BAND} raw")
    else:
        axes[0].text(0.5, 0.5, "raw cutout missing", transform=axes[0].transAxes, ha="center")
    if clipped:
        imshow_asinh(axes[1], fits.getdata(clipped), f"{PRIMARY_BAND} clipped")
    else:
        axes[1].text(0.5, 0.5, "clipped cutout missing", transform=axes[1].transAxes, ha="center")
    fig.tight_layout(); plt.show()


In [20]:
# Refit + overlay the selected isophote on the reference-band cutout, so we can SEE the PA we trust.
def fit_and_overlay(obj_id: str, band: str):
    raw, clipped = _find_band_fits(obj_id, band)
    img_path = clipped or raw
    if img_path is None:
        print(f"No FITS for {obj_id}/{band}"); return
    data = fits.getdata(img_path).astype(float)
    # If clipped: photutils dies on NaN. Mean-fill so we can at least plot an isophote here.
    if not np.isfinite(data).all():
        data = np.where(np.isfinite(data), data, np.nanmedian(data))
    try:
        from photutils.isophote import EllipseGeometry, Ellipse
    except ImportError:
        print("photutils not installed locally — `pip install photutils` to enable overlays."); return
    h, w = data.shape; cx, cy = w/2, h/2
    geom = EllipseGeometry(x0=cx, y0=cy, sma=max(w, h)/4, eps=0.3, pa=np.deg2rad(45))
    ellipse = Ellipse(data, geom)
    iso = ellipse.fit_image()
    if len(iso) == 0:
        print("isophote fit produced no rows."); return

    # Pull our recorded PA for this id/band from PA_results so we can compare.
    sel = pa_results[(pa_results["id"] == obj_id) & (pa_results["band"] == band)]
    pa_recorded = float(sel["est_pa"].iloc[0]) if len(sel) else np.nan
    sma_recorded = float(sel["est_sma"].iloc[0]) if len(sel) else np.nan

    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
    imshow_asinh(axes[0], data, f"{obj_id} · {band}")
    if np.isfinite(pa_recorded) and np.isfinite(sma_recorded):
        from matplotlib.patches import Ellipse as MPLEllipse
        eps = float(sel["est_ell"].iloc[0]) if "est_ell" in sel.columns else 0.3
        e = MPLEllipse((cx, cy), 2*sma_recorded, 2*sma_recorded*(1-eps),
                       angle=pa_recorded, fill=False, color="#e63946", lw=1.5)
        axes[0].add_patch(e)
        axes[0].set_title(f"{obj_id} · {band}  ·  PA={pa_recorded:.1f}°")

    # Radial profile
    sma_arr = np.array([i.sma for i in iso])
    int_arr = np.array([i.intens for i in iso])
    axes[1].plot(sma_arr, int_arr, color="#1d3557")
    axes[1].set_yscale("symlog"); axes[1].set_xlabel("sma [pix]"); axes[1].set_ylabel("intensity")
    axes[1].set_title("radial light profile"); axes[1].grid(alpha=0.3)
    fig.tight_layout(); plt.show()

if OBJ_ID:
    fit_and_overlay(OBJ_ID, PRIMARY_BAND)


In [21]:
# Optional: GALFIT model + residual for the same object.
def show_galfit_model(obj_id: str):
    obj_dir = LOCAL_GALFIT / obj_id
    if not obj_dir.exists():
        print(f"No GALFIT dir for {obj_id}"); return
    # GALFIT typically writes a multi-ext FITS: [0] = input, [1] = model, [2] = residual
    candidates = list(obj_dir.glob("*.fits"))
    if not candidates:
        print(f"No FITS in {obj_dir}"); return
    path = candidates[0]
    with fits.open(path) as hdul:
        if len(hdul) < 3:
            print(f"Unexpected GALFIT layout in {path.name}: {len(hdul)} HDUs"); return
        obs, mod, res = hdul[0].data, hdul[1].data, hdul[2].data
    fig, axes = plt.subplots(1, 3, figsize=(11, 4))
    imshow_asinh(axes[0], obs, "observed")
    imshow_asinh(axes[1], mod, "GALFIT model")
    imshow_asinh(axes[2], res, "residual")
    fig.tight_layout(); plt.show()

if OBJ_ID: show_galfit_model(OBJ_ID)


## 9 · Export figures + a one-row-per-object summary

Last cell builds `figures/<date>/` with the population-level plots we generated above (re-rendering each to a file), and writes `analysis_summary.csv` — one row per object with consensus PA, J-PLUS catalog PA, the residual, GALFIT params when present, and a `quality_flag`.

Drop both into your paper draft or hand them to a collaborator.


In [22]:
def save_summary_table() -> pd.DataFrame:
    """One row per object combining consensus PA, catalog PA, GALFIT params, and a flag."""
    if not len(pa_results): return pd.DataFrame()
    base = consensus.copy()
    if 'rec' in globals() and len(rec):
        base = base.merge(rec[["id","pa_jplus","circ_diff"]], on="id", how="left")
    if len(galfit):
        base = base.merge(galfit, on="id", how="left")
    # crude quality flag
    def flag(row):
        if pd.notna(row.get("galfit_chi2nu")) and row["galfit_chi2nu"] > 2:    return "galfit_bad"
        if pd.notna(row.get("circ_diff"))    and row["circ_diff"]    > 25:    return "pa_disagree"
        if pd.notna(row.get("circ_diff"))    and row["circ_diff"]    > PA_CONSENSUS_TOL_DEG: return "pa_warn"
        return "ok"
    base["quality_flag"] = base.apply(flag, axis=1)
    out = LOCAL_ROOT / "analysis_summary.csv"
    base.to_csv(out, index=False)
    print(f"[export] wrote {out}  ({len(base)} rows)")
    print(base["quality_flag"].value_counts())
    return base

summary = save_summary_table()
summary.head()


""


In [23]:
# Re-render the headline figures into figures/<today>/ for inclusion in a writeup.
import matplotlib
today_dir = LOCAL_FIGURES / datetime.now().strftime("%Y%m%d")
today_dir.mkdir(parents=True, exist_ok=True)

def _save(fig, name): fig.savefig(today_dir / name, dpi=150, bbox_inches="tight"); print("  ->", today_dir/name)

if len(pa_results):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(consensus["consensus_pa"].dropna(), bins=np.linspace(0, 180, 19),
            color="#264653", edgecolor="white")
    ax.set_xlabel("consensus PA [deg]"); ax.set_ylabel("objects"); ax.set_title("Consensus PA")
    _save(fig, "pa_consensus_hist.png"); plt.close(fig)

if 'rec' in globals() and len(rec):
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    ax.scatter(rec["pa_jplus"], rec["consensus_pa"], s=10, alpha=0.5, color="#1d3557")
    ax.plot([0,180],[0,180], color="grey", lw=0.6)
    ax.set_xlim(0,180); ax.set_ylim(0,180)
    ax.set_xlabel("pa_jplus [deg]"); ax.set_ylabel("consensus PA [deg]"); ax.set_title("PA reconciliation")
    _save(fig, "pa_reconciliation.png"); plt.close(fig)

if len(galfit):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(galfit["galfit_n"].clip(0, 8), bins=24, color="#264653", edgecolor="white")
    ax.set_xlabel("Sérsic n"); ax.set_ylabel("objects"); ax.set_title("Sérsic index distribution")
    _save(fig, "galfit_sersic_n.png"); plt.close(fig)

print(f"\nFigures + analysis_summary.csv ready under {LOCAL_FIGURES} and {LOCAL_ROOT}")



Figures + analysis_summary.csv ready under /Users/nacharya/Desktop/AntigravityProjects/palfitology/figures and /Users/nacharya/Desktop/AntigravityProjects/palfitology/cluster_pulls


## Appendix · Notes for the next pass

- **PSFs.** This notebook intentionally excludes `psfs_*/` from the sync (Section 1) — pull them only when you're studying band-dependent PSF effects on the morphology.
- **NaN-clipped cutouts + photutils.** Section 8's overlay mean-fills NaNs before handing data to `photutils.isophote`, because the fitter samples along contours and dies on NaN. This is *visualization only* — the pipeline's `--use-clipped-cutouts` still has the open caveat documented in `CONTEXT.md` / your memory `v0-6-fit-pa-clipped-consumption`.
- **V0.7 multi-source detection** will add a per-object flag for "this cutout has a neighbour." When that lands, add a `pa_results["neighbour_flag"]` cut to Section 5's consensus build.
- **GALFIT.** `src/palfitology/galfit.py` is still a stub. Until `palfitology fit-galfit` exists, Section 7 reads whatever externally-produced `fit.log` files happen to be in `LOCAL_GALFIT/<id>/`.
- **Reproducibility.** Re-running the whole notebook should be cheap once the sync has cached: `rsync` is incremental, table loads are pandas, plots redraw. The expensive step is the very first pull.


## 10 · Quick-pull: N summary PNGs from `fitted_pa_clipped_part_01`

Single cell, single knob. Set `N` to however many `*_summary.png` files you want from
`~/PALFITology_OLD/fitted_pa_clipped_part_01/all_summaries/` on the cluster,
re-run, done. Each run **rewrites** the local folder.

- Uses `rsync` over ssh via the `cefca` alias from your `~/.ssh/config`.
- Pure subprocess; no extra apps, no extra pip packages required.
- If `rsync` is missing the cell tells you `brew install rsync` and stops.


In [24]:
# === Cell 10 — pull N summary PNGs from fitted_pa_clipped_part_01/all_summaries =====
# ONE knob:
N = 500   # how many *_summary.png files to download. 5, 500, 20000 — whatever.

# -- everything below is fixed; do not touch ------------------------------------
import shutil, subprocess, sys
from pathlib import Path

# Reuse the CLUSTER_HOST / CLUSTER_USER you set in Section 0 (cell 2). Falls back
# to the known CEFCA target if this cell is run before Section 0.
try:
    _HOST_PART = f"{CLUSTER_USER}@{CLUSTER_HOST}" if CLUSTER_USER else CLUSTER_HOST
except NameError:
    _HOST_PART = "nacharya@hpc-login.office.cefca.es"

_REMOTE_DIR = "PALFITology_OLD/fitted_pa_clipped_part_01/all_summaries"
_LOCAL_DIR  = Path.home() / "Desktop/AntigravityProjects/palfitology/cluster_pulls/summary_pngs_part01"

print(f"remote target: {_HOST_PART}:{_REMOTE_DIR}")
print(f"local target : {_LOCAL_DIR}")
print(f"N requested  : {N}")

# 1. dependency check — rsync is the only thing we need outside Python stdlib.
if shutil.which("rsync") is None:
    sys.exit("rsync not found on PATH. Install with `brew install rsync` and re-run this cell.")

# 2. wipe-and-recreate local dir so every run is a clean overwrite.
if _LOCAL_DIR.exists():
    shutil.rmtree(_LOCAL_DIR)
_LOCAL_DIR.mkdir(parents=True, exist_ok=True)

# 3. ask the cluster for the first N *_summary.png filenames (lexical sort, deterministic).
print(f"\n[1/2] listing up to {N} summary PNGs on the cluster ...")
list_cmd = [
    "ssh", _HOST_PART,
    f"cd {_REMOTE_DIR} && ls -1 *_summary.png 2>/dev/null | sort | head -n {int(N)}",
]
res = subprocess.run(list_cmd, capture_output=True, text=True)
if res.returncode != 0:
    sys.exit(f"ssh listing failed:\n{res.stderr.strip()}")
names = [ln.strip() for ln in res.stdout.splitlines() if ln.strip()]
if not names:
    sys.exit("no *_summary.png files matched on the cluster — check the path.")
print(f"      cluster has {len(names)} matching file(s) (capped at N).")

# 4. write that filename list to a temp file and pull it in one rsync call.
list_file = _LOCAL_DIR / ".rsync_filelist.txt"
list_file.write_text("\n".join(names) + "\n")

print(f"\n[2/2] rsync-ing {len(names)} file(s) -> {_LOCAL_DIR} ...")
rsync_cmd = [
    "rsync", "-avh", "--progress",
    "--files-from", str(list_file),
    f"{_HOST_PART}:{_REMOTE_DIR}/",
    str(_LOCAL_DIR) + "/",
]
rc = subprocess.call(rsync_cmd)
list_file.unlink(missing_ok=True)

if rc != 0:
    sys.exit(f"rsync exited with code {rc}.")

got = sorted(_LOCAL_DIR.glob("*_summary.png"))
print(f"\nDone. {len(got)} / {N} requested PNGs in {_LOCAL_DIR}")


remote target: nacharya@hpc-login.office.cefca.es:PALFITology_OLD/fitted_pa_clipped_part_01/all_summaries
local target : /Users/nacharya/Desktop/AntigravityProjects/palfitology/cluster_pulls/summary_pngs_part01
N requested  : 500

[1/2] listing up to 500 summary PNGs on the cluster ...
      cluster has 500 matching file(s) (capped at N).

[2/2] rsync-ing 500 file(s) -> /Users/nacharya/Desktop/AntigravityProjects/palfitology/cluster_pulls/summary_pngs_part01 ...


** WARNING: connection is not using a post-quantum key exchange algorithm.
** This session may be vulnerable to "store now, decrypt later" attacks.
** The server may need to be upgraded. See https://openssh.com/pq.html


Transfer starting: 500 files
100707-11602_summary.png
          55686 100%    4.23MB/s   00:00:00 (xfer#1, to-check=0/500)
100707-13864_summary.png
          53215 100%    6.52MB/s   00:00:00 (xfer#2, to-check=1/500)
100707-17454_summary.png
          56148 100%    6.27MB/s   00:00:00 (xfer#3, to-check=2/500)
100707-21834_summary.png
          52912 100%    6.65MB/s   00:00:00 (xfer#4, to-check=3/500)
100707-25346_summary.png
          46596 100%    3.66MB/s   00:00:00 (xfer#5, to-check=4/500)
100707-2817_summary.png
          45621 100%    3.96MB/s   00:00:00 (xfer#6, to-check=5/500)
100707-28955_summary.png
          39031 100%   89.26MB/s   00:00:00 (xfer#7, to-check=6/500)
100707-31350_summary.png
          46252 100%    4.93MB/s   00:00:00 (xfer#8, to-check=7/500)
100707-31941_summary.png
          80978 100%    9.23MB/s   00:00:00 (xfer#9, to-check=8/500)
100707-32034_summary.png
          38262 100%   10.08MB/s   00:00:00 (xfer#10, to-check=9/500)
100707-33775_summary.png
      

## 11 · Build clipped-RGB cutouts on the cluster and download them

Same single-knob shape as Section 10. The cluster does the heavy lifting (it
already has the clipped FITS — no point dragging 3 × N FITS files to the laptop
just to compose them); we ship a tiny Python script over `ssh`, run it on the
cluster, then `rsync` the resulting PNGs down.

**Band choice.** R = iSDSS, G = rSDSS, B = gSDSS — the standard SDSS i/r/g Lupton
combination. iSDSS is the reddest broadband, gSDSS the bluest with good S/N, and
rSDSS anchors the mid-tone (it's also the band `detect.py` seeds geometry from).
That palette gives the canonical galaxy look: red bulges, blue-white arms.

**Inputs.** Per-object clipped FITS at
`~/PALFITology_old/images/<id>/clipped_cutouts_<ra>_<dec>/<band>_cutout.fits`.

**Output (local).**
`~/Desktop/AntigravityProjects/palfitology/cluster_pulls/rgb_clipped_images/<id>_rgb.png`.

The local folder is wiped on each run so it always matches the latest pull.


In [ ]:
# === Cell 11 — build N RGB PNGs on the cluster from clipped FITS, then pull =========
# ONE knob:
N = 200   # how many objects to render as RGB. 5, 200, 20000 — whatever.

# -- everything below is fixed; do not touch ------------------------------------
import shutil, subprocess, sys
from pathlib import Path
from string import Template

# Reuse Section 0\u2019s CLUSTER_HOST / CLUSTER_USER. Fallback to the known target.
try:
    _HOST_PART = f"{CLUSTER_USER}@{CLUSTER_HOST}" if CLUSTER_USER else CLUSTER_HOST
except NameError:
    _HOST_PART = "nacharya@hpc-login.office.cefca.es"

_REMOTE_DATA_ROOT = "PALFITology_OLD"
_REMOTE_IMAGES    = _REMOTE_DATA_ROOT + "/images"
_REMOTE_RGB_OUT   = _REMOTE_DATA_ROOT + "/rgb_clipped_images"

_LOCAL_DIR = Path.home() / "Desktop/AntigravityProjects/palfitology/cluster_pulls/rgb_clipped_images"

# Band assignment for the RGB.  R = iSDSS, G = rSDSS, B = gSDSS (SDSS i/r/g Lupton).
_R_BAND, _G_BAND, _B_BAND = "iSDSS", "rSDSS", "gSDSS"



# Cluster conda env that has astropy + matplotlib + photutils.
_REMOTE_CONDA_ENV = "palfitology"

# Shell prelude that *each* ssh command runs first, so conda activate sticks for
# the rest of THAT same command (every ssh call is otherwise a fresh shell).
# We try the four common conda install prefixes in order; whichever exists wins.
_CONDA_PRELUDE = (
    "for p in "
    "$HOME/miniconda3/etc/profile.d/conda.sh "
    "$HOME/anaconda3/etc/profile.d/conda.sh "
    "/opt/miniconda3/etc/profile.d/conda.sh "
    "/opt/anaconda3/etc/profile.d/conda.sh; "
    "do [ -f \"$p\" ] && source \"$p\" && break; done; "
    "conda activate " + _REMOTE_CONDA_ENV + " 2>/dev/null || { "
    "echo \"FATAL: could not source conda or activate env \u2018" + _REMOTE_CONDA_ENV + "\u2019\"; exit 9; }"
)

print(f"remote host  : {_HOST_PART}")
print(f"remote images: ~/{_REMOTE_IMAGES}")
print(f"remote rgb   : ~/{_REMOTE_RGB_OUT}")
print(f"local target : {_LOCAL_DIR}")
print(f"N requested  : {N}")
print(f"R,G,B bands  : ({_R_BAND}, {_G_BAND}, {_B_BAND})")
print(f"conda env    : {_REMOTE_CONDA_ENV}")

def _abort(msg: str) -> None:
    """Print loudly and stop the cell WITHOUT SystemExit (Jupyter hides
    everything above SystemExit in the traceback view)."""
    print("\n" + "=" * 60)
    print("ABORT: " + msg)
    print("=" * 60)
    raise RuntimeError(msg)

if shutil.which("rsync") is None:
    _abort("rsync not found on PATH. `brew install rsync` and re-run.")
if shutil.which("ssh") is None:
    _abort("ssh not found on PATH — cannot reach the cluster.")

# ---------------------------------------------------------------------------
# 0. Sanity probe.  EVERYTHING happens inside ONE ssh invocation, after the
#    conda prelude, so astropy/matplotlib resolve to the cluster\u2019s
#    palfitology env (not the base python).
# ---------------------------------------------------------------------------
print("\n[0/3] probing the cluster ...")
_PROBE_BODY = (
    "echo --- hostname ---; hostname; "
    "echo --- which python3 ---; which python3; python3 -V; "
    "echo --- astropy + matplotlib ---; "
    "python3 -c \"import astropy; from astropy.visualization import make_rgb, ManualInterval; "
    "import matplotlib; matplotlib.use(\\\"Agg\\\"); import matplotlib.pyplot; "
    "print(\\\"astropy\\\", astropy.__version__, \\\"matplotlib\\\", matplotlib.__version__)\"; "
    "echo --- images root ---; ls -d ~/" + _REMOTE_IMAGES + " 2>&1 | head -5; "
    "echo --- first 3 object dirs ---; ls -d ~/" + _REMOTE_IMAGES + "/*/ 2>/dev/null | head -3; "
    "echo --- one object\u2019s clipped subdir ---; "
    "first_obj=$(ls -d ~/" + _REMOTE_IMAGES + "/*/ 2>/dev/null | head -1); "
    "ls -d \"$first_obj\"clipped_cutouts_* 2>&1 | head -3; "
    "ls \"$first_obj\"clipped_cutouts_*/ 2>&1 | head -10"
)
probe_cmd = ["ssh", _HOST_PART, _CONDA_PRELUDE + "; " + _PROBE_BODY]
probe = subprocess.run(probe_cmd, capture_output=True, text=True)
print(probe.stdout)
if probe.stderr.strip():
    print("STDERR:\n" + probe.stderr)
if probe.returncode != 0:
    _abort(f"cluster probe failed with code {probe.returncode}. See output above.")

# ---------------------------------------------------------------------------
# 1. Wipe-and-recreate the local folder so each run is a clean overwrite.
# ---------------------------------------------------------------------------
if _LOCAL_DIR.exists():
    shutil.rmtree(_LOCAL_DIR)
_LOCAL_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# 2. Remote builder script.  string.Template ($placeholders) so the inner
#    Python can use { and } freely without colliding.
# ---------------------------------------------------------------------------
_REMOTE_TEMPLATE = Template(r"""
import os, sys, shutil, traceback
from pathlib import Path
import numpy as np
from astropy.io import fits

try:
    from astropy.visualization import make_rgb, ManualInterval
except Exception as e:
    print("FATAL: astropy.visualization.make_rgb / ManualInterval not available "
          "(needs astropy >= 6.1). Got:", e)
    sys.exit(2)

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except Exception as e:
    print("FATAL: matplotlib not available:", e)
    sys.exit(2)

HOME       = Path(os.path.expanduser("~"))
IMG_ROOT   = HOME / "$IMAGES"
OUT_ROOT   = HOME / "$OUTDIR"
N          = $N
R_BAND     = "$RBAND"
G_BAND     = "$GBAND"
B_BAND     = "$BBAND"

if OUT_ROOT.exists():
    shutil.rmtree(OUT_ROOT)
OUT_ROOT.mkdir(parents=True, exist_ok=True)

if not IMG_ROOT.exists():
    print("FATAL: image root does not exist:", IMG_ROOT)
    sys.exit(3)

obj_dirs = sorted([p for p in IMG_ROOT.iterdir() if p.is_dir()])
print("cluster: found", len(obj_dirs), "object dirs under", IMG_ROOT, "; will try the first", N)
if not obj_dirs:
    print("FATAL: no object subdirs under", IMG_ROOT)
    sys.exit(4)

def _read_band(clipped_dir, band):
    f = clipped_dir / (band + "_cutout.fits")
    if not f.exists():
        return None
    arr = fits.getdata(str(f)).astype(float)
    arr = np.where(np.isfinite(arr), arr, 0.0)
    return arr

made = 0
skipped = 0
first_err_printed = False
for obj_dir in obj_dirs:
    if made >= N:
        break
    try:
        cands = sorted(obj_dir.glob("clipped_cutouts_*"))
        if not cands:
            skipped += 1
            continue
        clipped_dir = cands[0]
        r = _read_band(clipped_dir, R_BAND)
        g = _read_band(clipped_dir, G_BAND)
        b = _read_band(clipped_dir, B_BAND)
        if r is None or g is None or b is None:
            skipped += 1
            continue
        h = min(r.shape[0], g.shape[0], b.shape[0])
        w = min(r.shape[1], g.shape[1], b.shape[1])
        r = r[:h, :w]; g = g[:h, :w]; b = b[:h, :w]
        # Joint 99.5th-percentile vmax across all three channels \u2014 one stretch
        # so faint outskirts and bright cores stay consistent between bands.
        pctl = 99.5
        vmax = 0.0
        for img in (r, g, b):
            v = float(np.percentile(img, pctl))
            if v > vmax:
                vmax = v
        if not (vmax > 0.0):
            skipped += 1
            continue
        # make_rgb expects (R, G, B) in that order. r,g,b locals already hold
        # iSDSS / rSDSS / gSDSS in (R,G,B) order.
        rgb = make_rgb(r, g, b, interval=ManualInterval(vmin=0.0, vmax=vmax))
        out_png = OUT_ROOT / (obj_dir.name + "_rgb.png")
        fig, ax = plt.subplots(figsize=(4, 4))
        ax.imshow(rgb, origin="lower")
        ax.set_axis_off()
        ax.set_title(obj_dir.name + "  R=" + R_BAND + " G=" + G_BAND + " B=" + B_BAND, fontsize=8)
        fig.tight_layout(pad=0.2)
        fig.savefig(str(out_png), dpi=140, bbox_inches="tight")
        plt.close(fig)
        made += 1
    except Exception as e:
        skipped += 1
        if not first_err_printed:
            print("first-object exception on", obj_dir.name, "->", repr(e))
            traceback.print_exc()
            first_err_printed = True

print("cluster: wrote", made, "RGB PNGs to", OUT_ROOT, " (skipped " + str(skipped) + ").")
if made == 0:
    print("FATAL: 0 RGB PNGs produced. Either the clipped_cutouts_* dirs do not have "
          "iSDSS/rSDSS/gSDSS files or every object hit an exception above.")
    sys.exit(5)
""")

_REMOTE_SCRIPT = _REMOTE_TEMPLATE.substitute(
    IMAGES=_REMOTE_IMAGES,
    OUTDIR=_REMOTE_RGB_OUT,
    N=str(int(N)),
    RBAND=_R_BAND,
    GBAND=_G_BAND,
    BBAND=_B_BAND,
)

import ast as _ast
try:
    _ast.parse(_REMOTE_SCRIPT)
except SyntaxError as e:
    print(_REMOTE_SCRIPT)
    _abort(f"rendered remote script has a syntax error locally: {e}")

print(f"\n[1/3] (script size: {len(_REMOTE_SCRIPT)} chars, parsed OK locally)")

# ---------------------------------------------------------------------------
# 3. Build on the cluster. Conda prelude + `python3 -u -` in ONE ssh call
#    so the activated env is alive when python starts. Script is piped on stdin.
# ---------------------------------------------------------------------------
print("\n[2/3] running RGB builder on the cluster (can take a minute for large N) ...")
remote_cmd = ["ssh", _HOST_PART, _CONDA_PRELUDE + "; python3 -u -"]
res = subprocess.run(remote_cmd, input=_REMOTE_SCRIPT, capture_output=True, text=True)
print("----- remote STDOUT -----")
print(res.stdout if res.stdout else "(empty)")
print("----- remote STDERR -----")
print(res.stderr if res.stderr else "(empty)")
print("----- end -----")
if res.returncode != 0:
    _abort(f"remote build script exited with code {res.returncode}. See STDERR above for the cause.")

# ---------------------------------------------------------------------------
# 4. rsync just the PNGs down. Small files, fast.
# ---------------------------------------------------------------------------
print(f"\n[3/3] rsync-ing PNGs -> {_LOCAL_DIR} ...")
rsync_cmd = [
    "rsync", "-avh", "--progress", "--delete",
    f"{_HOST_PART}:{_REMOTE_RGB_OUT}/",
    str(_LOCAL_DIR) + "/",
]
rc = subprocess.call(rsync_cmd)
if rc != 0:
    _abort(f"rsync exited with code {rc}.")

got = sorted(_LOCAL_DIR.glob("*_rgb.png"))
print(f"\nDone. {len(got)} RGB PNG(s) in {_LOCAL_DIR}")
